A [recording of this presentation](https://metoffice.sharepoint.com/:v:/r/sites/scienceitteam/Presentations/2026/Toy%20Build%20System%20Show%20_%20Tell-20260803_100140-Meeting%20Recording.mp4?d=w07d7a7ed116947b0bb2bc3058834e485&csf=1&web=1&e=IerhXb) is availabe.

# Toy Build System Development Update - August 2026

A quick demonstration of our development testing builds.

There is documentation which gives an [explination of the "work queue" concept](https://urban-adventure-kz4e1n7.pages.github.io/notes/work_queue.html).

## Preparation

Clone the repository, create a Python venv containing the toy and go to the developer tests...

In [ ]:
export WORK_DIR=$PWD/work
git clone ssh://git@github.com/MetOffice/ToyBuildSystem.git $WORK_DIR/ToyBuildSystem
cd $WORK_DIR/ToyBuildSystem
pip install .

## Deal With Libraries

The Toy makes use of the `pkgconfig` tool to locate 3rd party libraries. However some of those we use don't support this tool. In which case we can either specify the information on the command line or create our own `pkgconfig` files for them.

Both are demonstrated below.

In [ ]:
module load lfric
module unload py-sci-fab
export PKG_CONFIG_PATH=$PKG_CONFIG_PATH:~matthew.hambley/.local/lib/pkg_config
xios_exe=$(which xios_server.exe)
xios_bin=$(dirname "$xios_exe")
xios_inc_path=$(readlink -f "$xios_bin/../include")
xios_lib_path=$(readlink -f "$xios_bin/../lib")

export XIOS_ARG="-m xios:$xios_inc_path::$xios_lib_path"

Notice that there is no need to specify anything about NetCDF or YAXT as
they install their own `pkg_config` files.

## Building NEMO

NEMO is an older, and crustier, code base. In particular it has relied on magic within FCM to locate files for inclusion. It also relies on the linker's ability to find dependencies which interacts badly with our desire to only compile files which are used.

We handle these by making them explicit and user defined.

In [ ]:
cat $WORK_DIR/ToyBuildSystem/test/development/nemo.fabfile

In [ ]:
mkdir $WORK_DIR/nemo
cd $WORK_DIR/nemo
fabtoy $XIOS_ARG -v -j 4 -f $WORK_DIR/ToyBuildSystem/test/development/nemo.fabfile
ls -l work/bin

## Build Skeleton

Although a fairly straight-forward LFRic application build, it still
requires all the basics of any LFRic project, including PSyclone and
configuration.

In [ ]:
cat $WORK_DIR/ToyBuildSystem/test/development/skeleton.fabfile

In [ ]:
mkdir $WORK_DIR/skeleton
cd $WORK_DIR/skeleton
fabtoy $XIOS_ARG -v -j 4 -f $WORK_DIR/ToyBuildSystem/test/development/skeleton.fabfile
ls -l work/bin

## Comparison with Baf

One of the hoped-for gains in the toy work is to improve the user experience of using, but also, developing a build system.

In [ ]:
git clone ssh://git@github.com/MetOffice/lfric_core.git $WORK_DIR/lfric_core
cat $WORK_DIR/lfric_core/lfric_build/lfric_base.py

## Lets Build Atmosphere

The build scripts are just Python files so they can do all the things
a Python program can do. For instance, we might separate the list of
inclusions, when importing external models, into separate Python modules
and ``import`` them in the usual way.

For the moment the inclusion lists are just presented inline.

In [ ]:
cat $WORK_DIR/ToyBuildSystem/test/development/atmosphere.fabfile

In [ ]:
mkdir $WORK_DIR/atmosphere
cd $WORK_DIR/atmosphere
fabtoy $XIOS_ARG -v -j 4 -f $WORK_DIR/ToyBuildSystem/test/development/atmosphere.fabfile
ls -l work/bin

## Potential

Unlikely to see significant performance improvements due to significant
Amdahl fraction in, for instance, invoking a compiler, or PSyclone. See
[some performance profiles](https://urban-adventure-kz4e1n7.pages.github.io/notes/performance.html)
to illustrate this.

Instead the value comes from simplicity of implementation, leading to
better maintainability and extensibility.

Although it may seem that build systems are written and never revisited,
this is not so. Every time a new compiler is introduced, the build system
needs to be taught about it.

There are other potential examples of new tooling we may want to adopt
including profilers, coverage tools or alternatives for existing tools.

We may even want to adopt a new language for some projects and ideally
our build system framework would support this with minimal effort.

```python
from collections.abc import Iterable
from pathlib import Path

from fabengine.actions import Action, Prerequisite, SubprocessAction
from fabengine.actions.fortran import AnalyseFortran
from fabengine.artefact_store import ArtefactPath
from fabengine.database import Database
from fabengine.prerequisite import RealFilePrerequisite


class Templaterator(SubprocessAction):
    def __init__(
        self,
        filename: Path,
        leafname: ArtefactPath,
        inserts: dict[str, str],
        database: Database,
    ):
        self._filename = filename
        self._leafname = leafname
        self._database = database

        destination = database.artefact_store.get_artefact(leafname).real_filename
        arguments = ["-o", destination]
        for key, value in inserts.items():
            arguments.extend(["-s", key + "=" + value])
        arguments.append(self._filename)
        super().__init__(
            self._database.configuration["paths"]["templaterator"], arguments
        )

    def __str__(self) -> str:
        return f"Templating: {self._filename}"

    def prerequisites(self) -> Iterable[Prerequisite]:
        return [RealFilePrerequisite(self._filename)]

    def perform(self) -> list[Action]:
        self.run_subprocess()
        return [AnalyseFortran(self._leafname, self._database)]
```